In [2]:
# ============================================================================
# Token-Based Evaluation for Nepali Summarization Model
# ============================================================================

import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
from tqdm import tqdm
import sentencepiece as spm
import numpy as np

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

class EvalConfig:
    # Paths
    MODEL_CHECKPOINT = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\finetuned-summarization\best_model_old.pt"  # or epoch_X.pt
    TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
    TEST_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\test_norm.jsonl"
    OUTPUT_JSON = "evaluation_results.json"
    
    # Generation settings
    MAX_NEW_TOKENS = 64  # Reduced to match reference summary length
    GENERATION_TOP_K = 40
    GENERATION_TEMPERATURE = 0.7
    
    # Evaluation settings
    EVAL_SAMPLE_SIZE = None  # None = full test set, or set a number like 100

config = EvalConfig()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ============================================================================
# 2. MODEL ARCHITECTURE (Same as training script)
# ============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(input_ids)
        x = tok_emb + pos_emb
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        return loss, logits

# ============================================================================
# 3. LOAD MODEL & TOKENIZER
# ============================================================================

print("\n" + "="*80)
print("LOADING MODEL & TOKENIZER")
print("="*80)

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(config.TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab: {sp.vocab_size()})")

# Load fine-tuned model
checkpoint = torch.load(config.MODEL_CHECKPOINT, map_location=device, weights_only=False)
model_config = checkpoint['config']
model = GPT(model_config)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()
print(f"✓ Model loaded from: {os.path.basename(config.MODEL_CHECKPOINT)}")
if 'epoch' in checkpoint:
    print(f"  Epoch: {checkpoint['epoch']}, Step: {checkpoint.get('global_step', 'N/A')}")
if 'eval_loss' in checkpoint:
    print(f"  Eval Loss: {checkpoint['eval_loss']:.4f}")

# ============================================================================
# 4. GENERATION FUNCTION
# ============================================================================

PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"

def generate_summary_topk(model, tokenizer, text, max_new_tokens=64, top_k=40, temperature=0.7):
    """Generate summary using top-k sampling"""
    model.eval()
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated.size(1) >= model.config.block_size:
                break
            
            _, logits = model(generated)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            top_k_logits, top_k_indices = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1)
            probs = F.softmax(top_k_logits, dim=-1)
            next_token_idx = torch.multinomial(probs, 1)
            next_token = torch.gather(top_k_indices, -1, next_token_idx)
            
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop at EOS or padding token
            if next_token.item() == 0:
                break
    
    summary_tokens = generated[0, len(tokens):].tolist()
    summary = tokenizer.decode(summary_tokens)
    return summary, summary_tokens

# ============================================================================
# 5. TOKEN-BASED EVALUATION METRICS
# ============================================================================

def compute_token_overlap(ref_tokens, gen_tokens):
    """Compute number of overlapping tokens"""
    ref_set = set(ref_tokens)
    gen_set = set(gen_tokens)
    overlap = len(ref_set & gen_set)
    return overlap

def compute_precision(ref_tokens, gen_tokens):
    """Precision: % of generated tokens that appear in reference"""
    if len(gen_tokens) == 0:
        return 0.0
    
    ref_set = set(ref_tokens)
    correct = sum(1 for token in gen_tokens if token in ref_set)
    return correct / len(gen_tokens)

def compute_recall(ref_tokens, gen_tokens):
    """Recall: % of reference tokens that appear in generated"""
    if len(ref_tokens) == 0:
        return 0.0
    
    gen_set = set(gen_tokens)
    correct = sum(1 for token in ref_tokens if token in gen_set)
    return correct / len(ref_tokens)

def compute_f1(precision, recall):
    """F1 Score: Harmonic mean of precision and recall"""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def compute_token_edit_distance(ref_tokens, gen_tokens):
    """Levenshtein distance at token level"""
    m, n = len(ref_tokens), len(gen_tokens)
    
    # Create DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Initialize base cases
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    # Fill DP table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == gen_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )
    
    return dp[m][n]

def compute_token_similarity(ref_tokens, gen_tokens):
    """Normalized similarity based on edit distance"""
    edit_dist = compute_token_edit_distance(ref_tokens, gen_tokens)
    max_len = max(len(ref_tokens), len(gen_tokens))
    if max_len == 0:
        return 1.0
    return 1.0 - (edit_dist / max_len)

def evaluate_summary(ref_summary, gen_summary, tokenizer):
    """Compute all token-based metrics for a single summary pair"""
    # Tokenize both summaries
    ref_tokens = tokenizer.encode(ref_summary)
    gen_tokens = tokenizer.encode(gen_summary)
    
    # Compute metrics
    overlap = compute_token_overlap(ref_tokens, gen_tokens)
    precision = compute_precision(ref_tokens, gen_tokens)
    recall = compute_recall(ref_tokens, gen_tokens)
    f1 = compute_f1(precision, recall)
    edit_dist = compute_token_edit_distance(ref_tokens, gen_tokens)
    similarity = compute_token_similarity(ref_tokens, gen_tokens)
    
    return {
        'ref_token_count': len(ref_tokens),
        'gen_token_count': len(gen_tokens),
        'token_overlap': overlap,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'edit_distance': edit_dist,
        'similarity': similarity
    }

# ============================================================================
# 6. LOAD TEST DATA
# ============================================================================

print("\n" + "="*80)
print("LOADING TEST DATA")
print("="*80)

test_data = []
with open(config.TEST_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            if 'text' in item and 'summary' in item:
                test_data.append(item)

print(f"✓ Loaded {len(test_data)} test samples")

# Use subset if specified
if config.EVAL_SAMPLE_SIZE and config.EVAL_SAMPLE_SIZE < len(test_data):
    test_data = test_data[:config.EVAL_SAMPLE_SIZE]
    print(f"✓ Evaluating on {len(test_data)} samples")

# ============================================================================
# 7. RUN EVALUATION
# ============================================================================

print("\n" + "="*80)
print("RUNNING TOKEN-BASED EVALUATION")
print("="*80)

results = []
all_metrics = {
    'precision': [],
    'recall': [],
    'f1_score': [],
    'similarity': [],
    'edit_distance': [],
    'token_overlap': [],
    'ref_token_count': [],
    'gen_token_count': []
}

print(f"\nGenerating and evaluating summaries...")
for i, item in enumerate(tqdm(test_data, desc="Evaluating")):
    # Generate summary
    gen_summary, gen_tokens = generate_summary_topk(
        model, sp, item['text'],
        config.MAX_NEW_TOKENS,
        config.GENERATION_TOP_K,
        config.GENERATION_TEMPERATURE
    )
    
    # Evaluate
    metrics = evaluate_summary(item['summary'], gen_summary, sp)
    
    # Store individual result
    results.append({
        'sample_id': i,
        'article': item['text'][:200] + "...",  # First 200 chars
        'reference_summary': item['summary'],
        'generated_summary': gen_summary,
        'metrics': metrics
    })
    
    # Accumulate for averaging
    for key in all_metrics:
        all_metrics[key].append(metrics[key])

# ============================================================================
# 8. COMPUTE AGGREGATE STATISTICS
# ============================================================================

print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)

aggregate_stats = {}
for metric_name, values in all_metrics.items():
    aggregate_stats[metric_name] = {
        'mean': float(np.mean(values)),
        'std': float(np.std(values)),
        'min': float(np.min(values)),
        'max': float(np.max(values)),
        'median': float(np.median(values))
    }

# Print results
print(f"\n📊 AGGREGATE METRICS (n={len(test_data)} samples)")
print("="*80)

print(f"\n🎯 Quality Metrics:")
print(f"  Precision:       {aggregate_stats['precision']['mean']:.4f} ± {aggregate_stats['precision']['std']:.4f}")
print(f"  Recall:          {aggregate_stats['recall']['mean']:.4f} ± {aggregate_stats['recall']['std']:.4f}")
print(f"  F1 Score:        {aggregate_stats['f1_score']['mean']:.4f} ± {aggregate_stats['f1_score']['std']:.4f}")
print(f"  Similarity:      {aggregate_stats['similarity']['mean']:.4f} ± {aggregate_stats['similarity']['std']:.4f}")

print(f"\n📏 Length Statistics:")
print(f"  Ref tokens:      {aggregate_stats['ref_token_count']['mean']:.1f} ± {aggregate_stats['ref_token_count']['std']:.1f}")
print(f"  Gen tokens:      {aggregate_stats['gen_token_count']['mean']:.1f} ± {aggregate_stats['gen_token_count']['std']:.1f}")
print(f"  Token overlap:   {aggregate_stats['token_overlap']['mean']:.1f} ± {aggregate_stats['token_overlap']['std']:.1f}")
print(f"  Edit distance:   {aggregate_stats['edit_distance']['mean']:.1f} ± {aggregate_stats['edit_distance']['std']:.1f}")

# ============================================================================
# 9. SAVE RESULTS
# ============================================================================

output_data = {
    'evaluation_config': {
        'model_checkpoint': config.MODEL_CHECKPOINT,
        'test_data': config.TEST_JSONL,
        'num_samples': len(test_data),
        'max_new_tokens': config.MAX_NEW_TOKENS,
        'top_k': config.GENERATION_TOP_K,
        'temperature': config.GENERATION_TEMPERATURE
    },
    'aggregate_statistics': aggregate_stats,
    'individual_results': results
}

with open(config.OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Results saved to: {config.OUTPUT_JSON}")

# ============================================================================
# 10. SHOW SAMPLE OUTPUTS
# ============================================================================

print("\n" + "="*80)
print("SAMPLE OUTPUTS")
print("="*80)

# Show best, worst, and median samples based on F1 score
f1_scores = [r['metrics']['f1_score'] for r in results]
sorted_indices = np.argsort(f1_scores)

print(f"\n🏆 BEST SAMPLE (F1: {f1_scores[sorted_indices[-1]]:.4f}):")
print("─"*80)
best = results[sorted_indices[-1]]
print(f"Reference: {best['reference_summary']}")
print(f"Generated: {best['generated_summary']}")
print(f"Metrics: P={best['metrics']['precision']:.3f}, R={best['metrics']['recall']:.3f}, F1={best['metrics']['f1_score']:.3f}")

print(f"\n📊 MEDIAN SAMPLE (F1: {f1_scores[sorted_indices[len(sorted_indices)//2]]:.4f}):")
print("─"*80)
median = results[sorted_indices[len(sorted_indices)//2]]
print(f"Reference: {median['reference_summary']}")
print(f"Generated: {median['generated_summary']}")
print(f"Metrics: P={median['metrics']['precision']:.3f}, R={median['metrics']['recall']:.3f}, F1={median['metrics']['f1_score']:.3f}")

print(f"\n⚠️  WORST SAMPLE (F1: {f1_scores[sorted_indices[0]]:.4f}):")
print("─"*80)
worst = results[sorted_indices[0]]
print(f"Reference: {worst['reference_summary']}")
print(f"Generated: {worst['generated_summary']}")
print(f"Metrics: P={worst['metrics']['precision']:.3f}, R={worst['metrics']['recall']:.3f}, F1={worst['metrics']['f1_score']:.3f}")

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)
print(f"\n📁 Full results saved to: {config.OUTPUT_JSON}")
print(f"📊 Main metric (F1 Score): {aggregate_stats['f1_score']['mean']:.4f}")
print("="*80)

Using device: cuda

LOADING MODEL & TOKENIZER
✓ Tokenizer loaded (vocab: 16384)
✓ Model loaded from: best_model_old.pt
  Epoch: 5, Step: 5000
  Eval Loss: 2.9101

LOADING TEST DATA
✓ Loaded 858 test samples

RUNNING TOKEN-BASED EVALUATION

Generating and evaluating summaries...


Evaluating: 100%|██████████| 858/858 [05:56<00:00,  2.41it/s]


EVALUATION RESULTS

📊 AGGREGATE METRICS (n=858 samples)

🎯 Quality Metrics:
  Precision:       0.0502 ± 0.1059
  Recall:          0.0260 ± 0.0390
  F1 Score:        0.0281 ± 0.0444
  Similarity:      0.0072 ± 0.0153

📏 Length Statistics:
  Ref tokens:      27.5 ± 7.0
  Gen tokens:      24.5 ± 23.8
  Token overlap:   0.6 ± 0.9
  Edit distance:   36.8 ± 16.0

✓ Results saved to: evaluation_results.json

SAMPLE OUTPUTS

🏆 BEST SAMPLE (F1: 0.3030):
────────────────────────────────────────────────────────────────────────────────
Reference: कोभिड-१९ को खोप, खोप पासपोर्ट र कोभिड सङ्क्रमण नभएको पुष्टि गर्ने परीक्षणसम्बन्धी कागजपत्र "डार्कनेट"मा विक्री भइरहेको छ।
Generated: छ। ⁇ 
Metrics: P=1.000, R=0.179, F1=0.303

📊 MEDIAN SAMPLE (F1: 0.0000):
────────────────────────────────────────────────────────────────────────────────
Reference: कोरोनाभाइरस महामारी सुरु भएयता विश्वभरि यो रोगसँग जुध्न तीनवटा सामान्य विधि विश्व स्वास्थ्य सङ्गठन र धेरै देशका सरकारहरूले अघि सारिरहेका छन्।
Generated: पनि ⁇ 
